<a href="https://colab.research.google.com/github/group-geopulse/GeoPulse/blob/main/project_trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Using GDELT API via gdeltdoc

In [1]:
%pip install gdeltdoc

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gdeltdoc
from gdeltdoc import Filters
from datetime import datetime, timedelta

yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
day_before_yesterday = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')

f = Filters(
    start_date = day_before_yesterday,
    end_date = yesterday,
    domain = ['bloomberg.com', 'reuters.com']
    )

gd = gdeltdoc.GdeltDoc()

# Search for articles matching the filters
articles = gd.article_search(f)

Takes about 10 Seconds for 11/3 news, taken on 12/3.

In [3]:
articles.groupby(['domain']).count()

,url,url_mobile,title,seendate,socialimage,language,sourcecountry
domain,,,,,,,
bloomberg.com,11,11,11,11,11,11,11
jp.reuters.com,26,26,26,26,26,26,26


In [4]:
articles.head()

,url,url_mobile,title,seendate,socialimage,domain,language,sourcecountry
0,https://www.bloomberg.com/news/features/2025-0...,,Trump DOGE : Where the Musk Effort Is Restrict...,20250310T211500Z,https://assets.bwbx.io/images/users/iqjWHBFdfx...,bloomberg.com,English,United States
1,https://www.bloomberg.com/news/articles/2025-0...,,German Greens Propose Tighter Restrictions on ...,20250310T211500Z,https://assets.bwbx.io/images/users/iqjWHBFdfx...,bloomberg.com,English,United States
2,https://www.bloomberg.com/news/articles/2025-0...,,Selling of US Stocks by Systematic Funds Appea...,20250310T203000Z,https://assets.bwbx.io/images/users/iqjWHBFdfx...,bloomberg.com,English,United States
3,https://www.bloomberg.com/news/articles/2025-0...,,Venture Global ( VG ) Erases $38 Billion in Pa...,20250310T203000Z,https://assets.bwbx.io/images/users/iqjWHBFdfx...,bloomberg.com,English,United States
4,https://www.bloomberg.com/news/articles/2025-0...,,BBVA Gets Regulatory Approval to Offer Crypto ...,20250310T203000Z,https://assets.bwbx.io/images/users/iqjWHBFdfx...,bloomberg.com,English,United States


In [5]:
from datetime import datetime, timedelta
import pandas as pd

# Assuming 'articles' DataFrame is already loaded and contains the necessary columns

# Convert 'seendate' to 'Date' in the desired format
articles['seendate'] = pd.to_datetime(articles['seendate']).dt.strftime('%Y-%m-%d')
articles.rename(columns={'seendate': 'Date'}, inplace=True)

# Select the required columns and rename them
articles = articles[['Date', 'domain', 'title', 'url']]
articles.rename(columns={'domain': 'Source', 'title': 'Headline', 'url': 'Link'}, inplace=True)

# Display the first few rows of the transformed DataFrame
articles.head()

,Date,Source,Headline,Link
0,2025-03-10,bloomberg.com,Trump DOGE : Where the Musk Effort Is Restrict...,https://www.bloomberg.com/news/features/2025-0...
1,2025-03-10,bloomberg.com,German Greens Propose Tighter Restrictions on ...,https://www.bloomberg.com/news/articles/2025-0...
2,2025-03-10,bloomberg.com,Selling of US Stocks by Systematic Funds Appea...,https://www.bloomberg.com/news/articles/2025-0...
3,2025-03-10,bloomberg.com,Venture Global ( VG ) Erases $38 Billion in Pa...,https://www.bloomberg.com/news/articles/2025-0...
4,2025-03-10,bloomberg.com,BBVA Gets Regulatory Approval to Offer Crypto ...,https://www.bloomberg.com/news/articles/2025-0...


## Notes
- Does not return Tone or any other sentiment analysis score
- Does not appear to support Financial Times news - 'ft.com'
- Can't filter by language but seems to bias towards english anyways
- All news seems to be only from the US



## GDELT via BigQuery

In [ ]:
import pandas as pd

from google.cloud import bigquery
from google.colab import auth
from google.colab import drive

auth.authenticate_user()
drive.mount('drive')
client = bigquery.Client(project='gdelt-bq-4514')

In [ ]:
### Last line the query has been commented out as this can be done via the dataframe
### May need to uncomment to limit the search as attempting to retrieve a very large result will fail

##### Date, Source, Headline, Link, Snippet, Tone, Positive Score, Negative Score, Polarity

retrieval_query = (f'''
    SELECT
      Date,
      SourceCommonName as Source,
      REGEXP_EXTRACT(Extras, r'<PAGE_TITLE>(.*?)</PAGE_TITLE>') as Headline,
      DocumentIdentifier as Link,
      V2Tone
    FROM
      `gdelt-bq.gdeltv2.gkg`
    WHERE
      Date > 20100000000000 AND
      Date < 20200000000000 AND
      SourceCollectionIdentifier = 1 AND
      Extras LIKE '%<PAGE_TITLE>_%</PAGE_TITLE>%' AND
      TranslationInfo IS NULL AND
      SourceCommonName IN ('bloomberg.com', 'reuters.com', 'ft.com')
      -- LOWER(REGEXP_EXTRACT(Extras, r'<PAGE_TITLE>(.*?)</PAGE_TITLE>')) LIKE '%oil%'
  ''')


results = client.query(retrieval_query).to_dataframe()

### Export dataframe as CSV
Saved in personal Google Drive under Folder 'group-project'


In [ ]:
results.to_csv('gdelt_news_5_years.csv', index=False)
!cp gdelt_news_5_years.csv 'drive/My Drive/group-project'

### Load CSV from Google Drive

In [ ]:
results = pd.read_csv('drive/My Drive/group-project/gdelt_news_5_years.csv')

In [ ]:
### Extract data from column 'V2Tone' into relevant columns
results[['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Excess']] = results['V2Tone'].str.split(pat=',', n=4, expand=True)

### Drop column 'V2Tone' and 'Excess'
results = results.drop(columns=['V2Tone', 'Excess'])

In [ ]:
display(results)

In [8]:
# Checking if there are duplicate Headline in the dataset
duplicate_headlines = articles['Headline'].value_counts().loc[lambda x: x > 1]

if duplicate_headlines.empty:
    print("No duplicate headlines found.")
else:
    # Print the duplicate headlines and their counts
    print("Duplicate headlines and their counts:")
    print(duplicate_headlines)

    # Drop duplicate values
    articles = articles[~articles['Headline'].isin(duplicate_headlines.index)]

    # Print the number of articles after dropping duplicates
    print("Number of articles after dropping duplicates:", len(articles))

No duplicate headlines found.


In [9]:
### Extract rows with Healine that contain given keywords

# Total number of articles before filtering
total_articles = len(articles)
print(f"Total number of articles before filtering: {total_articles}")

# Keywords for filtering
keywords = ['tensions', 'crude', 'oil prices', 'oil supply', 'disruption', 'brent', 'sanctions', 'embargo', 'opec', 'middle east', 'russia', 'ukraine', 'petroleum', 'fuel', 'energy', 'climate', 'global warming']

# Adding regex pattern matching to each keyword string
for i in range(len(keywords)):
    keywords[i] = f'''( |[^a-z]|[^A-Z]){keywords[i]}( |[^a-z]|[^A-Z])'''

# Filter articles based on keywords
relevant_articles = articles[articles['Headline'].str.contains('|'.join(keywords), case=False, regex=True)]

# Number of relevant articles
num_relevant_articles = len(relevant_articles)
print(f"Number of relevant articles: {num_relevant_articles}")

# Number of irrelevant articles
num_irrelevant_articles = total_articles - num_relevant_articles
print(f"Number of irrelevant articles: {num_irrelevant_articles}")

# Update the articles DataFrame to only include relevant articles
articles = relevant_articles

# Print the first few rows of the filtered DataFrame
print("First few rows of the filtered articles:")
print(articles.head())

# for i in range(len(keywords)):
  #string = f'''LOWER(REGEXP_EXTRACT(Extras, r'<PAGE_TITLE>(.*?)</PAGE_TITLE>')) LIKE '{keywords[i]}' OR'''
  #print(string)

Total number of articles before filtering: 37
Number of relevant articles: 2
Number of irrelevant articles: 35
First few rows of the filtered articles:
          Date         Source  \
6   2025-03-10  bloomberg.com   
10  2025-03-10  bloomberg.com   

                                             Headline  \
6   COP30 Host Brazil Pitches Forest Funding as Cl...   
10  UK Starmer Plans Fresh Ukraine Peacekeeping Me...   

                                                 Link  
6   https://www.bloomberg.com/news/articles/2025-0...  
10  https://www.bloomberg.com/news/articles/2025-0...  


C:\Users\vindh\AppData\Local\Temp\ipykernel_42408\1536997788.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  relevant_articles = articles[articles['Headline'].str.contains('|'.join(keywords), case=False, regex=True)]


In [10]:
display(articles.info())

<class 'pandas.core.frame.DataFrame'>
Index: 2 entries, 6 to 10
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Date      2 non-null      object
 1   Source    2 non-null      object
 2   Headline  2 non-null      object
 3   Link      2 non-null      object
dtypes: object(4)
memory usage: 80.0+ bytes


None

In [11]:
articles.head()

,Date,Source,Headline,Link
6,2025-03-10,bloomberg.com,COP30 Host Brazil Pitches Forest Funding as Cl...,https://www.bloomberg.com/news/articles/2025-0...
10,2025-03-10,bloomberg.com,UK Starmer Plans Fresh Ukraine Peacekeeping Me...,https://www.bloomberg.com/news/articles/2025-0...


### Save & Load CSV from Google Drive

In [ ]:
filtered_results.to_csv('filtered_news_5_years.csv', index=False)
!cp filtered_news_5_years.csv 'drive/My Drive/group-project'

In [ ]:
filtered_results = pd.read_csv('drive/My Drive/group-project/filtered_news_5_years.csv')

In [ ]:
display(filtered_results)